In [2]:
import pandas as pd
import numpy as np

melb_df = pd.read_csv('data/melb_df_tuned3.csv', sep=';')
melb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Rooms          13580 non-null  int64  
 2   Type           13580 non-null  str    
 3   Price          13580 non-null  float64
 4   Method         13580 non-null  str    
 5   SellerG        13580 non-null  str    
 6   Date           13580 non-null  str    
 7   Distance       13580 non-null  float64
 8   Postcode       13580 non-null  int64  
 9   Bedroom        13580 non-null  int64  
 10  Bathroom       13580 non-null  int64  
 11  Car            13580 non-null  int64  
 12  Landsize       13580 non-null  float64
 13  BuildingArea   13580 non-null  float64
 14  CouncilArea    12211 non-null  str    
 15  Lattitude      13580 non-null  float64
 16  Longtitude     13580 non-null  float64
 17  Regionname     13580 non-null  str    
 18  Propertycount  13

In [3]:
# По всем столбцам
melb_df.groupby(by='Type').mean(numeric_only=True)

,Rooms,Price,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,Lattitude,Longtitude,Propertycount,MeanRoomsArea,AreaRatio,MonthSale,AgeBuilding,WeekdaySale,Weekend
Type,,,,,,,,,,,,,,,,,,
h,3.260874,1.242665e+06,10.979479,3104.080643,3.229336,1.613822,1.772674,617.181924,152.162553,-37.803795,144.994700,7259.025505,18.996731,-0.490031,7.121706,55.669700,4.846545,0.950259
t,2.837522,9.337351e+05,9.851346,3100.777379,2.814183,1.809695,1.555655,279.606822,134.649710,-37.815782,144.996489,7094.459605,18.569847,-0.094916,6.901257,26.690305,4.789946,0.937163
u,1.963871,6.051275e+05,7.607391,3110.797481,1.966523,1.183295,1.128936,477.314219,102.235863,-37.823710,144.996363,8199.280080,21.068242,0.319883,6.890620,39.703016,4.726218,0.927743


In [4]:
melb_df.groupby(by='Type')['Price'].mean()

Type
h    1.242665e+06
t    9.337351e+05
u    6.051275e+05
Name: Price, dtype: float64

In [42]:
# С несколькими аггрегациями
melb_df.groupby('Regionname')['Price']\
.agg(['count', 'mean', 'min', 'max', 'std'])\
.sort_values('count', ascending=False)

,count,mean,min,max,std
Regionname,,,,,
Southern Metropolitan,4695,1.372963e+06,131000.0,8000000.0,822529.158614
Northern Metropolitan,3890,8.981711e+05,145000.0,4525000.0,443967.693818
Western Metropolitan,2948,8.664205e+05,85000.0,3900000.0,394144.700287
Eastern Metropolitan,1471,1.104080e+06,288000.0,4000000.0,467417.860272
South-Eastern Metropolitan,450,9.229438e+05,266000.0,9000000.0,506525.336100
Eastern Victoria,53,6.999808e+05,400000.0,1085000.0,165498.190135
Northern Victoria,41,5.948293e+05,330000.0,1355000.0,217761.670461
Western Victoria,32,3.975234e+05,280500.0,710000.0,86211.956532


In [12]:
# количество комнат, для которых цена наибольшая
melb_df.groupby(by='Rooms')['Price'].mean()

Rooms
1     4.338245e+05
2     7.750812e+05
3     1.076081e+06
4     1.445282e+06
5     1.870260e+06
6     1.849366e+06
7     1.920700e+06
8     1.602750e+06
10    9.000000e+05
Name: Price, dtype: float64

In [13]:
# Какой регион итмеет наименьшее стандартное отклонение по географической широте (lattitude)
melb_df.groupby('Regionname')['Lattitude'].std().sort_values(ascending=True)

Regionname
Western Victoria              0.011579
Southern Metropolitan         0.043080
Eastern Metropolitan          0.047890
Northern Metropolitan         0.049639
Western Metropolitan          0.051251
South-Eastern Metropolitan    0.073411
Northern Victoria             0.084455
Eastern Victoria              0.147067
Name: Lattitude, dtype: float64

In [15]:
'''
Какая риелторская компания (свойство SellerG) имеет наименьшую общую выручку
за период с 1 мая по 1 сентября (включительно) 2017 года
'''
melb_df['Date'] = pd.to_datetime(melb_df['Date'])
melb_df_range = melb_df[
    (melb_df['Date'] >= '2017-05-01')
    & (melb_df['Date'] <= '2017-09-01')
]
melb_df_range.groupby('SellerG')['Price'].sum().sort_values(ascending=True)

SellerG
LITTLE             4509000.0
Burnham            5968000.0
Moonee             8045000.0
Bells             10311000.0
Thomson           10974000.0
Alexkarbon        11387000.0
Cayzer            15000000.0
Nick              16665000.0
Douglas           17586000.0
Rendina           18242276.0
Eview             19036500.0
McDonald          19727500.0
Buckingham        20597100.0
Collins           22730000.0
C21               23222500.0
Love              23357500.0
Philip            24895300.0
Williams          24996000.0
Purplebricks      26826501.0
HAR               28690000.0
Chisholm          29663250.0
RW                31751500.0
Village           32533000.0
Raine             34343700.0
O'Brien           36835508.0
Stockdale         42174800.0
Gary              46945300.0
Sweeney           47091250.0
Miles             51962000.0
YPA               59917350.0
Jas               61651100.0
Brad              63168000.0
Kay               64859500.0
Hodges            67881500.0
RT    

Построим сводную таблицу (pivot_table), вкоторой строками будут являться названия регионов,а столбцами признак выходного дня

In [19]:
melb_df.pivot_table(
    values='Price',
    index='Rooms',
    columns='Type',
    fill_value=0
).round()

Type,h,t,u
Rooms,,,
1,866866.0,592705.0,389929.0
2,1017238.0,710158.0,610491.0
3,1109233.0,984709.0,850596.0
4,1462283.0,1217092.0,1037476.0
5,1877327.0,1035000.0,0.0
6,1869508.0,0.0,520000.0
7,1920700.0,0.0,0.0
8,1510286.0,0.0,2250000.0
10,900000.0,0.0,0.0


In [24]:
# Как средняя и медианная площадь зависит от типа обьъекта и региона
o = melb_df.pivot_table(
    values='Landsize',
    index='Regionname',
    columns='Type',
    aggfunc=['median', 'mean'],
    fill_value=0
)
o

median                       mean              \
Type                            h      t      u            h           t   
Regionname                                                                 
Eastern Metropolitan        674.0  233.5  203.0   717.422847  269.440678   
Eastern Victoria            843.0    0.0  230.0  3108.960000    0.000000   
Northern Metropolitan       459.5  134.0    0.0   619.249092  317.325733   
Northern Victoria           724.0    0.0    0.0  3355.463415    0.000000   
South-Eastern Metropolitan  630.5  240.0  199.0   664.306701  212.160000   
Southern Metropolitan       586.0  246.0    0.0   569.643881  278.858824   
Western Metropolitan        531.0  198.0   62.0   507.883406  244.560669   
Western Victoria            599.5    0.0    0.0   655.500000    0.000000   

                                        
Type                                 u  
Regionname                              
Eastern Metropolitan        330.444444  
Eastern Victoria            295.333333  
Northern Metropolitan       495.026538  
Northern Victoria             0.000000  
South-Eastern Metropolitan  357.864865  
Southern Metropolitan       466.380245  
Western Metropolitan        557.637232  
Western Victoria              0.000000

In [25]:
o.columns

MultiIndex([('median', 'h'),
            ('median', 't'),
            ('median', 'u'),
            (  'mean', 'h'),
            (  'mean', 't'),
            (  'mean', 'u')],
           names=[None, 'Type'])

In [26]:
o['mean']['u']

Regionname
Eastern Metropolitan          330.444444
Eastern Victoria              295.333333
Northern Metropolitan         495.026538
Northern Victoria               0.000000
South-Eastern Metropolitan    357.864865
Southern Metropolitan         466.380245
Western Metropolitan          557.637232
Western Victoria                0.000000
Name: u, dtype: float64

In [28]:
mask = o['mean'] < o['median']
o[mask]

median                   mean            
Type                            h      t   u           h       t   u
Regionname                                                          
Eastern Metropolitan          NaN    NaN NaN         NaN     NaN NaN
Eastern Victoria              NaN    NaN NaN         NaN     NaN NaN
Northern Metropolitan         NaN    NaN NaN         NaN     NaN NaN
Northern Victoria             NaN    NaN NaN         NaN     NaN NaN
South-Eastern Metropolitan    NaN  240.0 NaN         NaN  212.16 NaN
Southern Metropolitan       586.0    NaN NaN  569.643881     NaN NaN
Western Metropolitan        531.0    NaN NaN  507.883406     NaN NaN
Western Victoria              NaN    NaN NaN         NaN     NaN NaN

In [ ]:
'''
Составим сводную таблицу, показывающую зависимость
медианной площади BuildingAREA от типа объекта и количества жилых комнат
'''
pivot = melb_df.pivot_table(
    values='BuildingArea',
    index='Rooms',
    columns='Type',
    aggfunc='median'
)
pivot

Type,h,t,u
Rooms,,,
1,126.0,88.0,69.5
2,126.0,114.0,110.0
3,126.0,126.0,126.0
4,141.0,159.5,126.0
5,177.0,152.0,NaN
6,126.0,NaN,171.0
7,216.5,NaN,NaN
8,126.0,NaN,126.0
10,126.0,NaN,NaN


In [34]:
series = pivot.stack()
series.idxmax()

(np.int64(7), 'h')

Составьте сводную таблицу, которая показывает зависимость медианной цены объекта недвижимости (Price) от риелторского агентства (SellerG) и типа здания (Type).

Во вновь созданной таблице найдите агентство, у которого медианная цена для зданий типа unit максимальна. В качестве ответа запишите название этого агентства.

In [38]:
pivot = melb_df.pivot_table(\
    values='Price',
    index='SellerG',
    columns='Type',
    aggfunc='median'
)


In [41]:
pivot['u'].sort_values(ascending=False)


SellerG
Nick             900000.0
Marshall         715000.0
Cayzer           707500.0
Kay              695000.0
Noel             693250.0
Buxton           670000.0
Fletchers        653000.0
Chisholm         640000.0
Philip           636000.0
RT               630000.0
C21              625500.0
RW               625500.0
Moonee           622000.0
O'Brien          618500.0
Hodges           605500.0
Eview            602000.0
Woodards         600000.0
Collins          592000.0
Jellis           591750.0
Purplebricks     582500.0
Williams         577000.0
hockingstuart    566000.0
Alexkarbon       560000.0
Miles            555000.0
Buckingham       553000.0
Gary             550000.0
Biggin           548000.0
Greg             547500.0
McGrath          542500.0
other            535000.0
Nelson           530000.0
HAR              518000.0
McDonald         502500.0
Barry            502000.0
Harcourts        500000.0
Ray              497500.0
Thomson          495000.0
Douglas          494500.0
Brad